In [ ]:
from datasets import load_from_disk
#dataset = load_from_disk("/scratch/project_2005092/nima/binary_dataset")
dataset = load_from_disk(r"C:\Users\alrazz\Documents\GitHub\persian_registers\binary_dataset") #change this with above for CSC
train_dataset = dataset["train"]
dev_dataset = dataset["validation"]
test_dataset = dataset["test"]

In [4]:
from datasets import concatenate_datasets

full_dataset = concatenate_datasets(
    [
        train_dataset,
        dev_dataset,
        test_dataset
    ]
)

print(full_dataset)

Dataset({
    features: ['u', 'id', 'ts', 'text', 'Turku_NLP', 'Turku_NLP_sub', 'Tags', 'web-register', 'Binary', 'source'],
    num_rows: 4472
})


In [5]:
print(full_dataset["Turku_NLP"][:5])
print(full_dataset["Turku_NLP_sub"][:5])
print(full_dataset["Binary"][:5])

['IP ; HI ; ID', 'IN ; IN', 'NA', 'IN ; SP', 'LY ; OP']
['oe ; oh ; -', 'lt ; oi', 'sr', 'dtp ; it', '- ; rs']
[1, 1, 1, 1, 1]


In [6]:
print(full_dataset[0])

{'u': 'http://elitemods.net/post/1478', 'id': 'ec64cb4634ad48717f53f4c59e98d812', 'ts': '2015-03-23T16:00:24Z', 'text': '|برای اولین بار در میان وبلاگ و وبسایت های ایرانی|\nمجموعه سلاح های بازی Battlefield 3 برای GTA IV به همراه پک صدا!\nحتما شما نام بازی بتلفیلد 3 رو شنیده اید و مطمئنا تریلرها و تصاویر این بازی را دیده اید و شاید هم بازی کرده اید. سلاح های واقعی و با گرافیک و صداهای طبیعی آن باعث شده با تا بازی زیبایی از آب درآید. حال شما دانلود و نصب این مد همین سلاح ها رو در بازی GTA IV خواهید دید و صداهای آنها واقعی تر خواهد شد! توصیه می کنم که این مجموعه بی نظیر را از دست ندهید.\nبرای دانلود و توضیحات بیشتر + آموزش نصب به ادامه مطلب مراجعه نمایید…\nبا نصب این مجموعه آیکون های اسلحه ها نیز تغییر خواهد کرد و حتی شدت ضربه آن ها تغییر میکند!\nسلاح های این مجموعه با همکاری طراحان زیر ساخته شده اند:\nac.amir\nshonenlex\nmotorsport71\nRollY\nDanham\nrkfrkfrkf\nSeph\ncoltmaster1911\nbutterhole\nDeathsDeciple\n47(Am-Team)\nNobeus\n» آموزش نصب:\nبرای نصب سلاح ها ابتدا فایل دانلود رو از حالت

In [7]:
labels_structure = {
    "MT": [],
    "LY": [],
    "SP": ["it", "os"],
    "ID": [],
    "NA": ["ne", "sr", "nb", "on"],
    "HI": ["re", "oh"],
    "IN": ["en", "ra", "dtp", "fi", "lt", "oi"],
    "OP": ["rv", "ob", "rs", "av", "oo"],
    "IP": ["ds", "ed", "oe"],
}


all_valid_labels = sorted(
    list(labels_structure.keys())
    + [
        s
        for subs in labels_structure.values()
        for s in subs
    ]
)


print(all_valid_labels)

['HI', 'ID', 'IN', 'IP', 'LY', 'MT', 'NA', 'OP', 'SP', 'av', 'ds', 'dtp', 'ed', 'en', 'fi', 'it', 'lt', 'nb', 'ne', 'ob', 'oe', 'oh', 'oi', 'on', 'oo', 'os', 'ra', 're', 'rs', 'rv', 'sr']


In [8]:
labels_to_remove = {
    "oi",
    "os",
    "on",
    "oh",
    "oo",
    "oe"
}

In [9]:
def extract_labels(row):

    labels = set()

    for column in ["Turku_NLP", "Turku_NLP_sub"]:

        value = row[column]

        if value:
            for label in value.split(";"):

                label = label.strip()

                # ignore empty values and "-"
                if label == "" or label == "-":
                    continue

                # remove "other" categories
                if label in labels_to_remove:
                    continue

                # keep only valid labels
                if label in all_valid_labels:
                    labels.add(label)


    return sorted(labels)

In [10]:
for i in range(5):
    print(
        full_dataset[i]["Turku_NLP"],
        "+"
        ,
        full_dataset[i]["Turku_NLP_sub"],
        "=>",
        extract_labels(full_dataset[i])
    )

IP ; HI ; ID + oe ; oh ; - => ['HI', 'ID', 'IP']
IN ; IN + lt ; oi => ['IN', 'lt']
NA + sr => ['NA', 'sr']
IN ; SP + dtp ; it => ['IN', 'SP', 'dtp', 'it']
LY ; OP + - ; rs => ['LY', 'OP', 'rs']


In [11]:
cleaned_labels = [
    extract_labels(row)
    for row in full_dataset
]

print(cleaned_labels[:5])

[['HI', 'ID', 'IP'], ['IN', 'lt'], ['NA', 'sr'], ['IN', 'SP', 'dtp', 'it'], ['LY', 'OP', 'rs']]


In [12]:
import numpy as np

label_to_index = {
    label: i
    for i, label in enumerate(all_valid_labels)
}


y = np.zeros(
    (len(cleaned_labels), len(all_valid_labels)),
    dtype=int
)


for row_idx, labels in enumerate(cleaned_labels):
    for label in labels:
        y[row_idx, label_to_index[label]] = 1

In [13]:
print(y.shape)
print(y[:5])

(4472, 31)
[[1 1 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1]
 [0 0 1 0 0 0 0 0 1 0 0 1 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0]]


In [14]:
texts = full_dataset["text"]

print(len(texts))
print(texts[0][:300])

4472
|برای اولین بار در میان وبلاگ و وبسایت های ایرانی|
مجموعه سلاح های بازی Battlefield 3 برای GTA IV به همراه پک صدا!
حتما شما نام بازی بتلفیلد 3 رو شنیده اید و مطمئنا تریلرها و تصاویر این بازی را دیده اید و شاید هم بازی کرده اید. سلاح های واقعی و با گرافیک و صداهای طبیعی آن باعث شده با تا بازی زیبایی 


In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(
    "BAAI/bge-m3-retromae"
)


X_emb = embedder.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
)

In [ ]:
print(X_emb.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import cross_val_predict


clf = OneVsRestClassifier(
    LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        n_jobs=-1,
    )
)


pred_probs = cross_val_predict(
    clf,
    X_emb,
    y,
    cv=5,
    method="predict_proba",
    n_jobs=-1,
)

In [ ]:
print(pred_probs.shape)

In [ ]:
labels_list = [
    [
        i
        for i, value in enumerate(row)
        if value == 1
    ]
    for row in y
]

In [ ]:
print(labels_list[:5])

In [ ]:
from cleanlab.multilabel_classification.filter import find_label_issues
from cleanlab.multilabel_classification.rank import get_label_quality_scores


issue_indices = find_label_issues(
    labels=labels_list,
    pred_probs=pred_probs,
    return_indices_ranked_by="self_confidence"
)


quality_scores = get_label_quality_scores(
    labels_list,
    pred_probs,
)


print(
    f"Found {len(issue_indices)} suspicious examples"
)

In [ ]:
import pandas as pd

issues_df = pd.DataFrame(
    {
        "index": issue_indices,
        "id": [
            full_dataset[i]["id"]
            for i in issue_indices
        ],
        "source": [
            full_dataset[i]["source"]
            for i in issue_indices
        ],
        "text": [
            full_dataset[i]["text"]
            for i in issue_indices
        ],
        "given_labels": [
            cleaned_labels[i]
            for i in issue_indices
        ],
        "quality_score": [
            quality_scores[i]
            for i in issue_indices
        ],
    }
)

In [ ]:
issues_df.head()

In [ ]:
def get_predicted_labels(idx, threshold=0.3):

    return [
        label
        for label, prob in zip(
            all_valid_labels,
            pred_probs[idx]
        )
        if prob >= threshold
    ]


issues_df["model_labels"] = [
    get_predicted_labels(i)
    for i in issue_indices
]


In [ ]:
def get_label_probabilities(idx):

    return {
        label: round(float(prob), 4)
        for label, prob in zip(
            all_valid_labels,
            pred_probs[idx]
        )
    }


issues_df["label_probabilities"] = [
    get_label_probabilities(i)
    for i in issue_indices
]


In [ ]:
issues_df = issues_df.sort_values(
    "quality_score"
)


In [ ]:
issues_df.to_csv(
    "cleanlab_annotation_review.csv",
    index=False,
    encoding="utf-8-sig"
)